# Лабораторная работа 2. Борьба со спамом
## Часть 1. Разбор писем и классификатор на основе чёрного списка

**Выполнили:**
  Симахин Степан Андреевич ИНБс-5301-01-00
  Игнатович Вадим Олегович ИНБс-5301-01-00

**Что делаем в этой части:**
1. Учимся читать файлы электронных писем (формат RFC 822) средствами Python.
2. Осваиваем базовую обработку текста: токенизацию, удаление стоп-слов, стемминг (библиотека **NLTK**).
3. Строим простейший классификатор спама — **«чёрный список слов»** — и честно измеряем его качество на тестовой выборке.

**Датасет:** [TREC 2007 Public Spam Corpus](https://plg.uwaterloo.ca/~gvcormac/treccorpus07/) — 10 000 реальных писем (подвыборка `trec07delay`), размеченных как `spam` / `ham` («не спам»).

> Блокнот рассчитан на **Google Colab** или локальный Jupyter. Ячейки выполняются сверху вниз: **Среда выполнения → Запустить все** (Runtime → Run all).

## Шаг 0. Подготовка датасета

**Если работаете локально** (Jupyter на своём ПК): папку `datasets` нужно поместить рядом с блокнотом.

**Если работаете в Google Colab:** файловая система Colab временная, поэтому датасет нужно загружать в каждой новой сессии. Выберите **один** из двух способов и раскомментируйте соответствующие строки в ячейке ниже:

- **Вариант А (рекомендуется) — через Google Диск.** Один раз загрузите файл `datasets.zip` в корень «Моего диска» (drive.google.com → «Создать» → «Загрузить файл»), затем в каждой сессии подключайте диск и распаковывайте архив.
- **Вариант Б — загрузка в сессию.** Загрузите `datasets.zip` напрямую в Colab (~40 МБ; при каждом перезапуске среды загрузку придётся повторять).

In [ ]:
# --- Google Colab, вариант А: архив лежит в корне Google Диска ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
#!unzip -q "https://drive.google.com/file/d/1463kbM03oDOlWuSTNBNfa-st8vWacUxv/" -P 1 -d /content/
!unzip -P 1 -q "/content/drive/MyDrive/lab2_datasets_with_pass.zip" -d /content/

Mounted at /content/drive
unzip:  cannot find or open /content/drive/MyDrive/lab2_datasets_with_pass.zip, /content/drive/MyDrive/lab2_datasets_with_pass.zip.zip or /content/drive/MyDrive/lab2_datasets_with_pass.zip.ZIP.


In [ ]:
import os

DATA_DIR = 'datasets/trec07delay/data/'           # каталог с письмами
LABELS_FILE = 'datasets/trec07delay/delay/index'  # файл с метками spam/ham

# Проверяем, что датасет на месте, прежде чем идти дальше
assert os.path.isdir(DATA_DIR), (
    'Папка datasets/trec07delay/data не найдена. '
    'Распакуйте datasets.zip рядом с блокнотом (см. шаг 0).')
assert os.path.isfile(LABELS_FILE), (
    'Файл меток datasets/trec07delay/delay/index не найден.')

print(f'Датасет на месте: {len(os.listdir(DATA_DIR))} файлов писем')

Датасет на месте: 10000 файлов писем


## Шаг 1. Как устроено электронное письмо

Каждый файл `inmail.N` — это письмо в классическом формате **RFC 822**: сначала идут **заголовки** (`From`, `To`, `Subject`, `Date`, `Received`…), затем **пустая строка**, затем **тело** письма. Тело может состоять из нескольких **MIME-частей** — например, `text/plain` (обычный текст) и `text/html` (то же содержимое в разметке HTML).

Посмотрим на «сырое» письмо глазами — первые 1500 символов файла:

In [ ]:
with open(DATA_DIR + 'inmail.2', encoding='utf-8', errors='ignore') as f:
    raw_email = f.read()

print(raw_email[:1500])

From bounce-debian-mirrors=ktwarwic=speedy.uwaterloo.ca@lists.debian.org  Sun Apr  8 13:09:29 2007
Return-Path: <bounce-debian-mirrors=ktwarwic=speedy.uwaterloo.ca@lists.debian.org>
Received: from murphy.debian.org (murphy.debian.org [70.103.162.31])
	by speedy.uwaterloo.ca (8.12.8/8.12.5) with ESMTP id l38H9S0I003031
	for <ktwarwic@speedy.uwaterloo.ca>; Sun, 8 Apr 2007 13:09:28 -0400
Received: from localhost (localhost [127.0.0.1])
	by murphy.debian.org (Postfix) with QMQP
	id 90C152E68E; Sun,  8 Apr 2007 12:09:05 -0500 (CDT)
Old-Return-Path: <yan.morin@savoirfairelinux.com>
X-Spam-Checker-Version: SpamAssassin 3.1.4 (2006-07-26) on murphy.debian.org
X-Spam-Level: 
X-Spam-Status: No, score=-1.1 required=4.0 tests=BAYES_05 autolearn=no 
	version=3.1.4
X-Original-To: debian-mirrors@lists.debian.org
Received: from xenon.savoirfairelinux.net (savoirfairelinux.net [199.243.85.90])
	by murphy.debian.org (Postfix) with ESMTP id 827432E3E5
	for <debian-mirrors@lists.debian.org>; Sun,  8 Apr 2

Видны заголовки `From`, `To`, `Subject`, `Date`, цепочка `Received` (по ней видно, через какие серверы прошло письмо) и начало тела.

Разбирать такой файл вручную, построчно — плохая идея: у формата много частных случаев (вложения, кодировки, многострочные заголовки). Воспользуемся стандартной библиотекой **`email`**, которая разбирает RFC 822 корректно:

In [ ]:
import email
from email import policy

with open(DATA_DIR + 'inmail.2', 'rb') as f:   # 'rb': письмо — байтовый файл, кодировки внутри бывают любые
    msg = email.message_from_binary_file(f, policy=policy.default)

print('From:   ', msg['From'])
print('To:     ', msg['To'])
print('Subject:', msg['Subject'])
print('Date:   ', msg['Date'])
print('Тип содержимого:', msg.get_content_type())

From:    Yan Morin <yan.morin@savoirfairelinux.com>
To:      debian-mirrors@lists.debian.org
Subject: Typo in /debian/README
Date:    Sun, 08 Apr 2007 12:52:30 -0400
Тип содержимого: text/plain


Теперь напишем функцию `text_email(path)`, которая достаёт из письма **тему и текстовую часть тела** (`text/plain`). HTML-части и вложения игнорируем — для анализа нужен чистый текст. Если письмо существует только в HTML (типичная история для спама), функция вернёт почти пустую строку — дальше такие письма мы пропустим и посчитаем, сколько их.

In [ ]:
def text_email(path):
    """Извлекает тему и текстовое (text/plain) тело письма из eml-файла."""
    try:
        with open(path, 'rb') as f:
            msg = email.message_from_binary_file(f, policy=policy.default)
        subject = str(msg.get('Subject') or '')
        parts = []
        for part in msg.walk():                    # обходим все MIME-части письма
            if part.get_content_type() == 'text/plain':
                try:
                    content = part.get_content()   # декодируем часть в строку
                    if isinstance(content, str):
                        parts.append(content)
                except Exception:
                    pass                           # битая кодировка — пропускаем часть
        return subject + ' ' + ' '.join(parts)
    except Exception:
        return ''                                  # совсем битый файл — пустая строка

# Проверим на том же письме
text = text_email(DATA_DIR + 'inmail.2')
print(text[:500])

Typo in /debian/README Hi, i've just updated from the gulus and I check on other mirrors.
It seems there is a little typo in /debian/README file

Example:
http://gulus.usherbrooke.ca/debian/README
ftp://ftp.fr.debian.org/debian/README

"Testing, or lenny.  Access this release through dists/testing.  The
current tested development snapshot is named etch.  Packages which
have been tested in unstable and passed automated tests propogate to
this release."

etch should be replace by lenny like in the


## Шаг 2. От текста к словам: токенизация, стоп-слова, стемминг

Чтобы компьютер мог сравнивать тексты, разобьём их на слова и приведём к «каноническому» виду. Три стандартные операции (все — из библиотеки **NLTK**):

1. **Токенизация** — разбиение текста на отдельные слова (токены): `'You have won!'` → `['you', 'have', 'won', '!']`.
2. **Удаление стоп-слов** — выбрасываем самые частотные слова языка (`the`, `is`, `are`…): они есть в любом тексте и ничего не говорят о его теме.
3. **Стемминг** — отсекаем окончания и суффиксы, оставляя основу: `congratulations` и `congrats` → `congrat`. Разные формы слова превращаются в один признак.

При первом запуске NLTK нужно один раз скачать словарные данные (несколько секунд):

In [ ]:
import nltk
nltk.download('punkt')      # модель токенизатора
nltk.download('punkt_tab')  # та же модель для новых версий NLTK (>= 3.8.2)
nltk.download('stopwords')  # списки стоп-слов для разных языков
print('Данные NLTK готовы к работе')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Данные NLTK готовы к работе


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [18]:
import string

PUNCTUATION = set(string.punctuation)   # !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
STOPWORDS = set(nltk.corpus.stopwords.words('english'))
STEMMER = nltk.PorterStemmer()

def stems_from_text(text):
    """Превращает текст в список основ слов (без стоп-слов и пунктуации)."""
    tokens = nltk.word_tokenize(text.lower())                 # 1. токенизация
    tokens = [t.strip(string.punctuation) for t in tokens]    # обрезаем пунктуацию по краям
    tokens = [t for t in tokens if t and t not in PUNCTUATION]  # 2. убираем мусорные токены
    tokens = [t for t in tokens if t not in STOPWORDS]          #    и стоп-слова
    return [STEMMER.stem(t) for t in tokens]                  # 3. стемминг

# Демонстрация на короткой фразе
demo = 'Congratulations! You have won the lottery, congratulations again!!!'
print(demo)
print('→', stems_from_text(demo))

# ...и на письме из датасета
print('→', stems_from_text(text_email(DATA_DIR + 'inmail.2'))[:20])

Congratulations! You have won the lottery, congratulations again!!!
→ ['congratul', 'lotteri', 'congratul']
→ ['typo', 'debian/readm', 'hi', 'updat', 'gulu', 'check', 'mirror', 'seem', 'littl', 'typo', 'debian/readm', 'file', 'exampl', 'http', 'gulus.usherbrooke.ca/debian/readm', 'ftp', 'ftp.fr.debian.org/debian/readm', 'test', 'lenni', 'access']


## Шаг 3. Метки и разбиение на тренировочную и тестовую выборки

Рядом с письмами лежит файл `delay/index` — в каждой его строке метка (`spam`/`ham`) и путь к файлу:

```
spam ../data/inmail.1
ham  ../data/inmail.2
...
```

Прочитаем его в словарь `labels`: *имя файла → метка*.

Дальше — важнейший шаг машинного обучения: **данные делятся на две непересекающиеся части**:

- **тренировочная выборка (70 %)** — по ней мы *строим* классификатор;
- **тестовая выборка (30 %)** — на ней мы *проверяем* его качество.

Тестовую выборку нельзя использовать при построении классификатора — иначе проверка превратится в самообман (все равно, что экзамен по заранее выданным билетам). Здесь берём простое детерминированное разбиение: первые 70 % файлов — в train, остальные — в test (в части 2 разбиение будет случайным).

In [20]:
from collections import Counter

# Читаем метки: имя файла → 'spam'/'ham'
labels = {}
with open(LABELS_FILE) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 2:
            label, path = parts
            labels[path.split('/')[-1]] = label

# Сортируем имена inmail.N по номеру N — разбиение будет одинаковым на любом компьютере
filelist = sorted(labels, key=lambda name: int(name.split('.')[1]))

TRAIN_RATIO = 0.7
train_files = filelist[:int(len(filelist) * TRAIN_RATIO)]
test_files  = filelist[int(len(filelist) * TRAIN_RATIO):]

print(f'Всего писем: {len(filelist)}  |  train: {len(train_files)}  |  test: {len(test_files)}')
print('train:', Counter(labels[f] for f in train_files))
print('test: ', Counter(labels[f] for f in test_files))

Всего писем: 10000  |  train: 7000  |  test: 3000
train: Counter({'spam': 5482, 'ham': 1518})
test:  Counter({'spam': 2549, 'ham': 451})


Обратите внимание: спама заметно больше, чем нормальных писем — **классы несбалансированы**. Запомним этот факт, он понадобится при интерпретации метрик.

## Шаг 4. Строим чёрный список спам-слов

Первая идея классификатора имитирует здравый смысл: у спама есть «характерные» слова (*lottery*, *viagra*, *winner*…), которые редко встречаются в нормальной переписке.

Алгоритм:
1. Собираем множество основ слов из **всех спам-писем** тренировочной выборки → `spam_words`.
2. То же для нормальных писем → `ham_words`.
3. **Чёрный список** = слова, которые встречаются в спаме, но **ни разу** — в нормальных письмах: `spam_words − ham_words` (разность множеств).

Цикл по 7000 писем занимает примерно полминуты — просто подождите.

In [21]:
spam_words = set()
ham_words = set()
skipped = 0

for filename in train_files:
    stems = stems_from_text(text_email(DATA_DIR + filename))
    if len(stems) <= 2:        # слишком короткое или HTML-only письмо — пропускаем
        skipped += 1
        continue
    if labels[filename] == 'spam':
        spam_words.update(stems)
    else:
        ham_words.update(stems)

blacklist = spam_words - ham_words

print(f'Различных слов в спам-письмах: {len(spam_words)}, в обычных: {len(ham_words)}')
print(f'Пропущено «нечитаемых» писем из train: {skipped}')
print(f'Чёрный список: {len(blacklist)} слов')

Различных слов в спам-письмах: 23133, в обычных: 28764
Пропущено «нечитаемых» писем из train: 354
Чёрный список: 16794 слов


In [22]:
# Заглянём внутрь чёрного списка: 30 случайных слов
import random
random.seed(42)
print(random.sample(sorted(blacklist), 30))

['coton', '37340', 'ofcours', 'medaryvil', 'lain', 'easyier', 'claw', 'bushmen', 'willingli', '53857457', '49.95window', 'cassidi', 'kimballton', 'long-lash', '�����p�����', '399.00', "i'll��h", 'whichand', 'know.kick', 'прoдaвцoв', 'oneaspect', '124231', 'flewisign-pro', 'winki', 'serenad', 'on.gam', 'felton', 'just��', 'scorn', 'clarklik']


Среди слов видны и «настоящие» спам-маркеры, и мусор: обрывки URL, имена файлов, обрезки слов в экзотических кодировках. Для первой прикидки это приемлемо — но запомним: **качество признаков напрямую влияет на качество классификатора**.

## Шаг 5. Классификация тестовой выборки и матрица ошибок

Правило классификации предельно простое: **если в письме есть хотя бы одно слово из чёрного списка — это спам.**

Прогоняем правило по тестовой выборке и раскладываем результаты по четырём "корзинам":

| | прогноз: спам | прогноз: ham |
|---|---|---|
| **на самом деле спам** | **TP** — верно пойман | **FN** — спам пропущен |
| **на самом деле ham** | **FP** — ложная тревога | **TN** — верно пропущен |

Эта таблица называется **матрицей ошибок** (confusion matrix). Для почтового фильтра хуже всего **FP**: пользователь потеряет настоящее письмо.

In [23]:
tp = fp = tn = fn = 0
skipped_test = 0

for filename in test_files:
    stems = set(stems_from_text(text_email(DATA_DIR + filename)))
    if len(stems) <= 2:
        skipped_test += 1
        continue
    predicted = 'spam' if stems & blacklist else 'ham'  # пересечение с чёрным списком непусто?
    actual = labels[filename]
    if predicted == 'spam' and actual == 'spam':
        tp += 1
    elif predicted == 'spam' and actual == 'ham':
        fp += 1
    elif predicted == 'ham' and actual == 'spam':
        fn += 1
    else:
        tn += 1

import pandas as pd
confusion = pd.DataFrame(
    [[tp, fn],
     [fp, tn]],
    index=['на самом деле спам', 'на самом деле ham'],
    columns=['прогноз: спам', 'прогноз: ham'])
print(f'Пропущено «нечитаемых» писем из test: {skipped_test}')
confusion

Пропущено «нечитаемых» писем из test: 143


,прогноз: спам,прогноз: ham
на самом деле спам,1184,1226
на самом деле ham,195,252


In [25]:
total = tp + fp + tn + fn
accuracy  = (tp + tn) / total                               # доля верных ответов
precision = tp / (tp + fp) if (tp + fp) else 0              # чистота: сколько из помеченных спамом — действительно спам
recall    = tp / (tp + fn) if (tp + fn) else 0              # полнота: какую долю спама поймали
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print(f'Классифицировано писем: {total}')
print(f'Accuracy  (доля верных):       {accuracy:.1%}')
print(f'Precision (чистота по спаму):  {precision:.1%}')
print(f'Recall    (полнота по спаму):  {recall:.1%}')
print(f'F1-мера:                       {f1:.1%}')
print()
naive = (tp + fn) / total   # доля настоящего спама в тесте
print(f'Для сравнения: бессмысленное правило «любое письмо — спам» дало бы accuracy {naive:.1%}')

Классифицировано писем: 2857
Accuracy  (доля верных):       50.3%
Precision (чистота по спаму):  85.9%
Recall    (полнота по спаму):  49.1%
F1-мера:                       62.5%

Для сравнения: бессмысленное правило «любое письмо — спам» дало бы accuracy 84.4%


## Выводы по части 1
1. Метрика accuracy чёрного списка (доля верных ответов) оказалось ниже, чем у бессмысленного правила "всё это спам", так как из множества потенциально спам-триггеров было предварительно вычтено множество потенциально не-спам-триггеров, из-за чего было увеличена общая доля верных ответов, но стала ниже по спаму.
2. Потому что обучение по тестировочной дате превратится в "самообман".
3. Использование замен символов на другие (замена "казино" на "ка3ин0"), использование нетекстового формата сообщения (HTML, изображения), использование метофарического описания (замена прямого по смыслу текста на переносные смысла и устойчивые фразы).
